In [ ]:
import numpy as  np
import cv2

In [ ]:
#connect to drive
from google.colab import drive
drive.mount('/content/drive')
from google.colab.patches import cv2_imshow

path = "/content/drive/MyDrive/Personal/Yolo/code/images/testing/"
model_folder = "/content/drive/MyDrive/Personal/Yolo/model yolo3/"

#get the webcam video stream
file_video_stream = cv2.VideoCapture(path + "video_sample2.mp4")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
#create a while loop
while (file_video_stream.isOpened()):
    #get the current frame from video stream
    ret,current_frame = file_video_stream.read()
    #use the video current frame instead of image
    img_to_detect = current_frame

    img_height = img_to_detect.shape[0]
    img_width = img_to_detect.shape[1]

    # convert to blob to pass into model
    img_blob = cv2.dnn.blobFromImage(img_to_detect, 0.003922, (416, 416), swapRB=True, crop=False)
    #recommended by yolo authors, scale factor is 0.003922=1/255, width,height of blob is 320,320
    #accepted sizes are 320×320,416×416,609×609. More size means more accuracy but less speed

    # set of 80 class labels
    class_labels = ["person","bicycle","car","motorcycle","airplane","bus","train","truck","boat",
                    "trafficlight","firehydrant","stopsign","parkingmeter","bench","bird","cat",
                    "dog","horse","sheep","cow","elephant","bear","zebra","giraffe","backpack",
                    "umbrella","handbag","tie","suitcase","frisbee","skis","snowboard","sportsball",
                    "kite","baseballbat","baseballglove","skateboard","surfboard","tennisracket",
                    "bottle","wineglass","cup","fork","knife","spoon","bowl","banana","apple",
                    "sandwich","orange","broccoli","carrot","hotdog","pizza","donut","cake","chair",
                    "sofa","pottedplant","bed","diningtable","toilet","tvmonitor","laptop","mouse",
                    "remote","keyboard","cellphone","microwave","oven","toaster","sink","refrigerator",
                    "book","clock","vase","scissors","teddybear","hairdrier","toothbrush"]

    #Declare List of colors as an array
    #Green, Blue, Red, cyan, yellow, purple
    #Split based on ',' and for every split, change type to int
    #convert that to a numpy array to apply color mask to the image numpy array
    class_colors = ["0,255,0","0,0,255","255,0,0","255,255,0","0,255,255"]
    class_colors = [np.array(every_color.split(",")).astype("int") for every_color in class_colors]
    class_colors = np.array(class_colors)
    class_colors = np.tile(class_colors,(16,1))

    # Loading pretrained model
    # input preprocessed blob into model and pass through the model
    # obtain the detection predictions by the model using forward() method
    model3_path = "J:/My Drive/Personal/Yolo/model yolo3/"
    yolo_model = cv2.dnn.readNetFromDarknet(model_folder+'yolov3.cfg',model_folder+'yolov3.weights')

    # Obtener nombres de las capas y normalizar índices de salida (compatible con todas las versiones)
    yolo_layers = yolo_model.getLayerNames()
    output_layer_indices = yolo_model.getUnconnectedOutLayers()

    # Normalizar a lista de ints (soporta scalar, array 2D, array 1D, list, tuple)
    if isinstance(output_layer_indices, np.ndarray):
        output_layer_indices = output_layer_indices.flatten().tolist()
    elif isinstance(output_layer_indices, (list, tuple)):
        output_layer_indices = list(output_layer_indices)
    else:
        output_layer_indices = [int(output_layer_indices)]

    output_layer_indices = [int(i) for i in output_layer_indices]  # asegurar enteros

    yolo_output_layer = [yolo_layers[i - 1] for i in output_layer_indices]

    # input preprocessed blob into model and pass through the model
    yolo_model.setInput(img_blob)
    # obtain the detection layers by forwarding through till the output layer
    obj_detection_layers = yolo_model.forward(yolo_output_layer)

    ############## NMS Change 1 ###############
    # initialization for non-max suppression (NMS)
    # declare list for [class id], [box center, width & height[], [confidences]
    class_ids_list = []
    boxes_list = []
    confidences_list = []
    ############## NMS Change 1 END ###########

    # loop over each of the layer outputs
    for object_detection_layer in obj_detection_layers:
      # loop over the detections
        for object_detection in object_detection_layer:

            # obj_detections[1 to 4] => will have the two center points, box width and box height
            # obj_detections[5] => will have scores for all objects within bounding box
            all_scores = object_detection[5:]
            predicted_class_id = np.argmax(all_scores)
            prediction_confidence = all_scores[predicted_class_id]

            # take only predictions with confidence more than 20%
            if prediction_confidence > 0.80:
                #get the predicted label
                predicted_class_label = class_labels[predicted_class_id]
                #obtain the bounding box co-oridnates for actual image from resized image size
                bounding_box = object_detection[0:4] * np.array([img_width, img_height, img_width, img_height])
                (box_center_x_pt, box_center_y_pt, box_width, box_height) = bounding_box.astype("int")
                start_x_pt = int(box_center_x_pt - (box_width / 2))
                start_y_pt = int(box_center_y_pt - (box_height / 2))

                ############## NMS Change 2 ###############
                #save class id, start x, y, width & height, confidences in a list for nms processing
                #make sure to pass confidence as float and width and height as integers
                class_ids_list.append(predicted_class_id)
                confidences_list.append(float(prediction_confidence))
                boxes_list.append([start_x_pt, start_y_pt, int(box_width), int(box_height)])
                ############## NMS Change 2 END ###########

    ############## NMS Change 3 ###############
    # Applying the NMS will return only the selected max value ids while suppressing the non maximum (weak) overlapping bounding boxes
    # Non-Maxima Suppression confidence set as 0.5 & max_suppression threhold for NMS as 0.4 (adjust and try for better perfomance)
    max_value_ids = cv2.dnn.NMSBoxes(boxes_list, confidences_list, 0.5, 0.4)

    # Defensive handling:
    if max_value_ids is None or len(max_value_ids) == 0:
        print("No boxes after NMS")
    else:
        # max_value_ids puede ser lista de ints, ndarray (N,1) o ndarray (N,)
        # Convertir todo a 1D array de indices
        import numpy as np
        if isinstance(max_value_ids, (list, tuple)):
            idxs = np.array(max_value_ids).reshape(-1)
        else:
            # es numpy array; aplanar y convertir a 1D
            idxs = max_value_ids.flatten()

        # iterar por indices
        for i in idxs:
            # i puede ser numpy.int64; convertir a int por seguridad
            i = int(i)
            box = boxes_list[i]
            start_x_pt = int(box[0])
            start_y_pt = int(box[1])
            box_width = int(box[2])
            box_height = int(box[3])

            predicted_class_id = class_ids_list[i]
            predicted_class_label = class_labels[predicted_class_id]
            prediction_confidence = confidences_list[i]

            end_x_pt = start_x_pt + box_width
            end_y_pt = start_y_pt + box_height

            box_color = class_colors[predicted_class_id]
            box_color = [int(c) for c in box_color]

            label_text = "{}: {:.2f}%".format(predicted_class_label, prediction_confidence * 100)
            print("predicted object {}".format(label_text))

            cv2.rectangle(img_to_detect, (start_x_pt, start_y_pt), (end_x_pt, end_y_pt), box_color, 1)
            cv2.putText(img_to_detect, label_text, (start_x_pt, start_y_pt-5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, box_color, 1)

    # show output
    cv2_imshow(img_to_detect)

    #terminate while loop if 'q' key is pressed
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

#releasing the stream and the camera
#close all opencv windows
file_video_stream.release()
cv2.destroyAllWindows()